# 01 — ETL Pipeline: De JSON crudo a tablas limpias

**Objetivo:** Transformar los 5.036 archivos JSON de Polar en tablas organizadas (DataFrames) que podamos analizar.

**ETL** significa:
- **E**xtract — leer los archivos desde el disco
- **T**ransform — limpiar, filtrar, reorganizar
- **L**oad — guardar en un formato cómodo (CSV)

Al final de este notebook tendrás 4 archivos CSV listos:
1. `sessions.csv` — una fila por sesión de entrenamiento
2. `fitness_tests.csv` — resultados de los tests de VO2max
3. `nightly_recovery.csv` — HRV y recovery por noche
4. `sleep_scores.csv` — puntuación de sueño por noche

---

## Paso 0: Preparar el entorno

Primero necesitamos **importar** las librerías. Esto es como abrir la caja de herramientas antes de trabajar.

- `json` — para leer archivos JSON (viene con Python, no hay que instalar nada)
- `glob` — para buscar archivos por patrón (ej: todos los que empiecen con 'training-session-')
- `os` — para manejar rutas de archivos
- `pandas` — la librería estrella para manejar tablas de datos
- `zipfile` — para descomprimir el ZIP de Polar

In [1]:
# Importar librerías
# Cada línea trae una herramienta distinta

import json          # Leer archivos JSON
import glob          # Buscar archivos por patrón
import os            # Manejar rutas y carpetas
import zipfile       # Descomprimir ZIP
import pandas as pd  # Tablas de datos (lo llamamos 'pd' por convención)
import numpy as np   # Cálculos numéricos (lo llamamos 'np')
from datetime import datetime  # Trabajar con fechas

# Esta línea hace que pandas muestre todas las columnas sin cortar
pd.set_option('display.max_columns', None)

print('Librerías cargadas correctamente ✓')

Librerías cargadas correctamente ✓


## Paso 1: Descomprimir el archivo de Polar

El export de Polar viene como un ZIP con miles de archivos JSON adentro.
Vamos a descomprimirlo en la carpeta `data/raw/`.

**IMPORTANTE:** Cambia la ruta en la siguiente celda para que apunte a donde está tu archivo ZIP.

In [2]:
# ============================================================
# CAMBIA ESTA RUTA para que apunte a tu archivo ZIP de Polar
# ============================================================
ZIP_PATH = '../polar-user-data-export_fd7e5202-2b19-4496-b54e-a95d3646e8d7.zip'

# Carpeta donde se van a descomprimir los archivos
RAW_DIR = '../data/raw'

# Carpeta donde guardaremos los CSV limpios
PROCESSED_DIR = '../data/processed'

# Crear las carpetas si no existen
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Descomprimir
# Solo lo hace si la carpeta raw está vacía (para no repetir)
if len(os.listdir(RAW_DIR)) == 0:
    print(f'Descomprimiendo {ZIP_PATH}...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(RAW_DIR)
    print(f'Listo. Archivos extraídos: {len(os.listdir(RAW_DIR))}')
else:
    print(f'La carpeta ya tiene {len(os.listdir(RAW_DIR))} archivos. No se descomprime de nuevo.')

La carpeta ya tiene 5036 archivos. No se descomprime de nuevo.


## Paso 2: Explorar qué hay adentro

Antes de transformar nada, miremos qué tipos de archivos tenemos.
Cada archivo tiene un prefijo que indica qué es: `training-session-`, `fitness-test-results-`, etc.

In [3]:
# Listar todos los archivos JSON
all_files = glob.glob(os.path.join(RAW_DIR, '*.json'))
print(f'Total de archivos JSON: {len(all_files)}')

# Contar por tipo (prefijo antes del primer guión con fecha)
# Ejemplo: 'training-session-2024-01-15T...' → prefijo = 'training-session'
from collections import Counter

prefixes = []
for f in all_files:
    name = os.path.basename(f)  # Solo el nombre, sin la ruta
    # Buscar el patrón: todo antes de la primera fecha (4 dígitos seguidos)
    parts = name.split('-')
    prefix_parts = []
    for part in parts:
        if part.isdigit() and len(part) == 4:  # Es un año (2019, 2020, etc.)
            break
        prefix_parts.append(part)
    prefixes.append('-'.join(prefix_parts))

prefix_counts = Counter(prefixes)
print('\nArchivos por tipo:')
for prefix, count in prefix_counts.most_common():
    print(f'  {prefix}: {count}')

Total de archivos JSON: 5036

Archivos por tipo:
  activity: 2442
  training-session: 1089
  training-target: 98
  fitness-test-results-43386190: 21
  generic-period-0d442000: 6
  generic-period-0da62000: 3
  generic-period-0dda4000: 3
  generic-period-0debc000: 3
  generic-period-0e06a000: 3
  generic-period-0da04000: 2
  generic-period-0da18000: 2
  generic-period-0da1e000: 2
  generic-period-0da28000: 2
  generic-period-0da2e000: 2
  generic-period-0da38000: 2
  generic-period-0da46000: 2
  generic-period-0da48000: 2
  generic-period-0da4c000: 2
  generic-period-0da52000: 2
  generic-period-0da78000: 2
  generic-period-0da7c000: 2
  generic-period-0da82000: 2
  generic-period-0da86000: 2
  generic-period-0da88000: 2
  generic-period-0da8a000: 2
  generic-period-0da90000: 2
  generic-period-0da96000: 2
  generic-period-0da98000: 2
  generic-period-0da9e000: 2
  generic-period-0daa4000: 2
  generic-period-0daaa000: 2
  generic-period-0daac000: 2
  generic-period-0dab4000: 2
  generic-

## Paso 3: Extraer sesiones de entrenamiento

Esta es la tabla más importante. Cada archivo `training-session-*.json` contiene:
- Fecha y hora de inicio/fin
- Duración, distancia, calorías
- FC promedio y máxima
- Training load (cardio, muscular, percibido)
- Deporte

Vamos a leer cada archivo y extraer los campos que nos interesan.

In [4]:
# Diccionario de IDs de deporte → nombre legible
# Polar usa IDs numéricos internamente
SPORT_NAMES = {
    '1': 'Running',
    '2': 'Cycling',
    '3': 'Walking',
    '12': 'Swimming',
    '15': 'Other indoor',
    '17': 'Hiking',
    '19': 'Road running',
    '23': 'Trail running',
    '36': 'Mountain biking',
    '38': 'Road cycling',
    '43': 'Strength training',
    '47': 'Core',
    '66': 'Group exercise',
    '83': 'Yoga',
    '126': 'Functional training',
    '186': 'Ultimate',
}

print(f'Diccionario de deportes cargado: {len(SPORT_NAMES)} deportes')

Diccionario de deportes cargado: 16 deportes


In [5]:
# Leer todas las sesiones de entrenamiento
session_files = sorted(glob.glob(os.path.join(RAW_DIR, 'training-session-*.json')))
print(f'Archivos de sesión encontrados: {len(session_files)}')

# Lista donde vamos a guardar cada sesión como un diccionario
sessions_list = []

for filepath in session_files:
    # Leer el archivo JSON
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    # Extraer el ID del deporte
    exercises = data.get('exercises', [])
    sport_id = ''
    if exercises and isinstance(exercises[0].get('sport'), dict):
        sport_id = str(exercises[0]['sport'].get('id', ''))
    
    # Extraer training load
    load_report = data.get('trainingLoadReport', {})
    
    # Construir un diccionario con los campos que nos interesan
    session = {
        'date': data.get('startTime', '')[:10],           # Solo la fecha
        'start_time': data.get('startTime', ''),          # Fecha + hora completa
        'duration_min': data.get('durationMillis', 0) / 60000,  # Milisegundos → minutos
        'sport_id': sport_id,
        'sport_name': SPORT_NAMES.get(sport_id, data.get('name', 'Unknown')),
        'distance_km': (data.get('distanceMeters', 0) or 0) / 1000,
        'calories': data.get('calories', 0),
        'hr_avg': data.get('hrAvg', None),
        'hr_max': data.get('hrMax', None),
        'cardio_load': load_report.get('cardioLoad', None),
        'muscle_load': load_report.get('muscleLoad', None),
        'perceived_load': load_report.get('perceivedLoad', None),
        'recovery_time_hours': float(data.get('recoveryTimeMillis', 0) or 0) / 3600000,
    }
    
    sessions_list.append(session)

# Convertir la lista de diccionarios a un DataFrame (tabla)
df_sessions = pd.DataFrame(sessions_list)

# Convertir la columna 'date' a tipo fecha (para poder ordenar y filtrar por fecha)
df_sessions['date'] = pd.to_datetime(df_sessions['date'])

# Ordenar por fecha
df_sessions = df_sessions.sort_values('date').reset_index(drop=True)

print(f'\nDataFrame creado: {len(df_sessions)} sesiones')
print(f'Columnas: {list(df_sessions.columns)}')
print(f'Rango de fechas: {df_sessions["date"].min().date()} → {df_sessions["date"].max().date()}')

Archivos de sesión encontrados: 1089

DataFrame creado: 1089 sesiones
Columnas: ['date', 'start_time', 'duration_min', 'sport_id', 'sport_name', 'distance_km', 'calories', 'hr_avg', 'hr_max', 'cardio_load', 'muscle_load', 'perceived_load', 'recovery_time_hours']
Rango de fechas: 2019-04-24 → 2026-03-23


In [6]:
# Veamos las primeras 10 filas
# .head(10) muestra las primeras 10 filas del DataFrame
df_sessions.head(10)

,date,start_time,duration_min,sport_id,sport_name,distance_km,calories,hr_avg,hr_max,cardio_load,muscle_load,perceived_load,recovery_time_hours
0,2019-04-24,2019-04-24T06:38:35,87.000000,3,Walking,3.361600,517,108,138,49.104000,-1.0,NaN,5.550000
1,2019-04-27,2019-04-27T09:27:39,503.395833,3,Walking,17.002301,2473,111,168,218.752000,-1.0,NaN,26.833333
2,2019-05-02,2019-05-02T19:21:24,101.720833,1,Running,9.147700,861,126,156,91.722700,-1.0,NaN,12.500000
3,2019-05-05,2019-05-05T23:55:35,14.097917,3,Walking,0.008100,17,60,72,0.368315,-1.0,NaN,0.016667
4,2019-05-06,2019-05-06T08:45:37,44.139583,3,Walking,0.199200,64,67,108,3.248920,-1.0,NaN,0.050000
5,2019-05-09,2019-05-09T19:50:44,65.881250,1,Running,4.111300,483,116,137,42.287500,-1.0,NaN,5.183333
6,2019-05-13,2019-05-13T20:19:25,128.302083,1,Running,9.600000,1149,129,162,119.454000,-1.0,NaN,17.533333
7,2019-05-16,2019-05-16T19:41:30,109.095833,1,Running,5.587200,762,113,134,66.657400,-1.0,NaN,8.083333
8,2019-05-20,2019-05-20T20:14:28,117.497917,1,Running,9.439500,1063,130,155,113.547000,-1.0,NaN,16.550000
9,2019-05-26,2019-05-26T08:25:38,266.252083,1,Running,0.695000,414,116,137,NaN,-1.0,NaN,1.616667


In [7]:
# Resumen estadístico de las columnas numéricas
# .describe() te da: count, mean, std, min, 25%, 50%, 75%, max
df_sessions.describe().round(1)

,date,duration_min,distance_km,calories,hr_avg,hr_max,cardio_load,muscle_load,perceived_load,recovery_time_hours
count,1089,1089.0,1089.0,1089.0,1089.0,1089.0,1081.0,1089.0,202.0,1089.0
mean,2023-03-16 06:14:12.892562,93.1,5.6,738.1,117.3,155.8,73.7,-1.0,424.4,11.4
min,2019-04-24 00:00:00,4.2,0.0,15.0,58.0,72.0,0.4,-1.0,8.9,0.0
25%,2021-12-08 00:00:00,53.9,0.0,358.0,106.0,141.0,32.2,-1.0,149.8,2.7
50%,2023-07-12 00:00:00,81.4,1.0,582.0,117.0,157.0,55.4,-1.0,332.0,6.2
75%,2024-07-14 00:00:00,118.0,4.8,997.0,128.0,171.0,100.2,-1.0,523.9,12.6
max,2026-03-23 00:00:00,546.8,435.1,3925.0,176.0,211.0,383.5,-1.0,2743.3,200.8
std,NaN,63.6,22.5,540.2,18.8,20.2,58.8,0.0,392.4,18.7


## Paso 4: Extraer tests de fitness (VO2max)

Los archivos `fitness-test-results-*.json` contienen los resultados del Polar Fitness Test,
que estima tu VO2max (OwnIndex).

In [8]:
# Leer tests de fitness
test_files = sorted(glob.glob(os.path.join(RAW_DIR, 'fitness-test-results-*.json')))
print(f'Archivos de fitness test encontrados: {len(test_files)}')

tests_list = []

for filepath in test_files:
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    result = data.get('fitnessTestResult', {})
    phys = result.get('physicalInformation', {})
    
    test = {
        'date': data.get('startTime', '')[:10],
        'vo2max_own_index': result.get('ownIndex', None),
        'fitness_class': result.get('fitnessClass', None),
        'hr_avg_test': result.get('averageHeartRate', None),
        'weight_kg': phys.get('weight', None),
        'hr_max_setting': phys.get('maximumHeartRate', None),
    }
    
    tests_list.append(test)

df_tests = pd.DataFrame(tests_list)
df_tests['date'] = pd.to_datetime(df_tests['date'])
df_tests = df_tests.sort_values('date').reset_index(drop=True)

print(f'DataFrame creado: {len(df_tests)} tests')
df_tests

Archivos de fitness test encontrados: 21
DataFrame creado: 21 tests


,date,vo2max_own_index,fitness_class,hr_avg_test,weight_kg,hr_max_setting
0,2022-04-02,70,ELITE,58,64.0,194
1,2022-05-03,64,ELITE,62,66.0,194
2,2022-07-13,65,ELITE,70,66.0,194
3,2022-08-12,68,ELITE,60,66.0,194
4,2023-03-01,63,ELITE,74,66.0,194
5,2023-05-07,68,ELITE,59,66.0,194
6,2023-06-21,69,ELITE,55,66.0,194
7,2023-08-28,68,ELITE,63,66.0,194
8,2024-01-26,66,ELITE,67,66.0,194
9,2024-04-02,68,ELITE,58,68.0,194


## Paso 5: Extraer datos de recovery nocturno (HRV)

El archivo `nightly_recovery_*.json` contiene métricas de HRV nocturno,
incluyendo RMSSD (la medida más usada en sport science para monitoreo de recovery).

In [9]:
# Leer nightly recovery
recovery_files = glob.glob(os.path.join(RAW_DIR, 'nightly_recovery_4*.json'))
print(f'Archivos de recovery encontrados: {len(recovery_files)}')

recovery_list = []

for filepath in recovery_files:
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    # Este archivo es una lista de noches (no un solo registro)
    if isinstance(data, list):
        entries = data
    else:
        entries = [data]
    
    for entry in entries:
        rri = entry.get('meanNightlyRecoveryRri', None)
        recovery = {
            'night': entry.get('night', ''),
            'rmssd': entry.get('meanNightlyRecoveryRmssd', None),
            'rri_ms': rri,
            'hr_rest': round(60000 / rri, 1) if rri and rri > 0 else None,
            'recovery_indicator': entry.get('recoveryIndicator', None),
            'baseline_rmssd': entry.get('meanBaselineRmssd', None),
            'baseline_rri': entry.get('meanBaselineRri', None),
        }
        recovery_list.append(recovery)

df_recovery = pd.DataFrame(recovery_list)
df_recovery['night'] = pd.to_datetime(df_recovery['night'])
df_recovery = df_recovery.sort_values('night').reset_index(drop=True)

print(f'DataFrame creado: {len(df_recovery)} noches')
print(f'Rango: {df_recovery["night"].min().date()} → {df_recovery["night"].max().date()}')
df_recovery.tail(10)

Archivos de recovery encontrados: 1
DataFrame creado: 1309 noches
Rango: 2022-03-20 → 2026-04-10


,night,rmssd,rri_ms,hr_rest,recovery_indicator,baseline_rmssd,baseline_rri
1299,2026-04-01,73.0,1124,53.4,5.0,70.0,1145.0
1300,2026-04-02,58.0,1080,55.6,2.0,70.0,1144.0
1301,2026-04-03,63.0,1096,54.7,1.0,70.0,1141.0
1302,2026-04-04,62.0,1102,54.4,1.0,69.0,1134.0
1303,2026-04-05,49.0,1003,59.8,2.0,69.0,1132.0
1304,2026-04-06,60.0,1071,56.0,2.0,69.0,1137.0
1305,2026-04-07,57.0,1113,53.9,4.0,68.0,1132.0
1306,2026-04-08,60.0,1143,52.5,4.0,67.0,1131.0
1307,2026-04-09,64.0,1052,57.0,2.0,66.0,1132.0
1308,2026-04-10,56.0,1045,57.4,2.0,66.0,1122.0


## Paso 6: Extraer sleep scores

In [10]:
# Leer sleep scores (corregido con las claves reales)
sleep_files = glob.glob(os.path.join(RAW_DIR, 'sleep_score_*.json'))
print(f'Archivos de sleep score encontrados: {len(sleep_files)}')

sleep_list = []

for filepath in sleep_files:
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    if isinstance(data, list):
        entries = data
    else:
        entries = [data]
    
    for entry in entries:
        score_result = entry.get('sleepScoreResult', {})
        sleep = {
            'night': entry.get('night', ''),
            'sleep_score': score_result.get('sleepScore', None),
            'continuity': score_result.get('continuityScore', None),
            'efficiency': score_result.get('efficiencyScore', None),
            'duration_score': score_result.get('groupDurationScore', None),
            'refresh_score': score_result.get('groupRefreshScore', None),
            'solidity_score': score_result.get('groupSolidityScore', None),
            'rem_score': score_result.get('remScore', None),
            'deep_sleep_score': score_result.get('n3Score', None),
        }
        sleep_list.append(sleep)

df_sleep = pd.DataFrame(sleep_list)
df_sleep['night'] = pd.to_datetime(df_sleep['night'])
df_sleep = df_sleep.sort_values('night').reset_index(drop=True)

print(f'DataFrame creado: {len(df_sleep)} noches')
print(f'Rango: {df_sleep["night"].min().date()} → {df_sleep["night"].max().date()}')
df_sleep.tail(10)

Archivos de sleep score encontrados: 1
DataFrame creado: 1323 noches
Rango: 2022-03-19 → 2026-04-10


,night,sleep_score,continuity,efficiency,duration_score,refresh_score,solidity_score,rem_score,deep_sleep_score
1313,2026-04-01,85.5243,76.0,80.6095,100.0000,78.5302,80.5365,92.1174,64.9429
1314,2026-04-02,76.1690,50.0,66.6074,100.0000,70.2877,64.2025,86.4067,54.1687
1315,2026-04-03,70.3613,64.0,74.9039,52.7500,79.3126,76.1346,83.5338,75.0913
1316,2026-04-04,42.1499,54.0,45.7450,20.6875,56.2146,47.0817,70.8295,41.5997
1317,2026-04-05,79.2318,58.0,77.9476,100.0000,71.3376,70.6492,97.0064,45.6688
1318,2026-04-06,77.9616,70.0,77.4983,100.0000,58.1165,76.4994,73.8945,42.3385
1319,2026-04-07,82.5742,68.0,82.0378,87.1667,80.5739,80.8459,97.2489,63.8989
1320,2026-04-08,73.1868,68.0,75.7422,73.4583,66.0743,77.7474,86.6398,45.5088
1321,2026-04-09,69.6928,66.0,73.0233,70.8333,58.0797,76.6744,61.9070,54.2525
1322,2026-04-10,83.7794,70.0,81.7968,85.1250,86.7044,80.9323,98.9109,74.4979


In [11]:
# Abrir un archivo de sleep y ver qué claves tiene realmente
import pprint  # "pretty print" — muestra diccionarios de forma legible

with open(sleep_files[0], 'r') as f:
    data = json.load(f)

# Tomar la última noche como ejemplo
last_entry = data[-1] if isinstance(data, list) else data

print("Claves del registro:")
pprint.pprint(list(last_entry.keys()))

print("\nClaves dentro de sleepScoreResult:")
score_result = last_entry.get('sleepScoreResult', {})
pprint.pprint(score_result)

Claves del registro:
['night', 'sleepScoreResult', 'sleepScoreBaselines']

Claves dentro de sleepScoreResult:
{'continuityScore': 70.0,
 'efficiencyScore': 81.7968,
 'groupDurationScore': 85.125,
 'groupRefreshScore': 86.7044,
 'groupSolidityScore': 80.9323,
 'longInterruptionsScore': 91.0,
 'n3Score': 74.4979,
 'remScore': 98.9109,
 'scoreRate': 4,
 'sleepScore': 83.7794,
 'sleepTimeOwnTargetScore': 83.0,
 'sleepTimeRecommendationScore': 87.25}


## Paso 7: Guardar como CSV

CSV (Comma-Separated Values) es el formato más universal para datos tabulares.
Lo puede abrir Excel, Google Sheets, cualquier lenguaje de programación.
Guardamos los 4 DataFrames en la carpeta `data/processed/`.

In [12]:
# Guardar cada DataFrame como CSV
# index=False evita guardar el número de fila como columna extra

df_sessions.to_csv(os.path.join(PROCESSED_DIR, 'sessions.csv'), index=False)
df_tests.to_csv(os.path.join(PROCESSED_DIR, 'fitness_tests.csv'), index=False)
df_recovery.to_csv(os.path.join(PROCESSED_DIR, 'nightly_recovery.csv'), index=False)
df_sleep.to_csv(os.path.join(PROCESSED_DIR, 'sleep_scores.csv'), index=False)

print('Archivos guardados en data/processed/:')
for f in os.listdir(PROCESSED_DIR):
    size_kb = os.path.getsize(os.path.join(PROCESSED_DIR, f)) / 1024
    print(f'  {f} — {size_kb:.0f} KB')

print('\n¡Pipeline ETL completado! ✓')

Archivos guardados en data/processed/:
  fitness_tests.csv — 1 KB
  nightly_recovery.csv — 55 KB
  sessions.csv — 122 KB
  sleep_scores.csv — 92 KB

¡Pipeline ETL completado! ✓


## Paso 8: Primera mirada rápida (EDA mínimo)

Antes de cerrar, hagamos unas visualizaciones básicas para confirmar que todo está bien.

In [13]:
import plotly.express as px

# Sesiones por deporte (top 10)
sport_counts = df_sessions['sport_name'].value_counts().head(10)

fig = px.bar(
    x=sport_counts.values,
    y=sport_counts.index,
    orientation='h',
    title='Sesiones por deporte (top 10)',
    labels={'x': 'Número de sesiones', 'y': 'Deporte'},
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=400)
fig.show()

In [14]:
# Sesiones por mes a lo largo del tiempo
df_sessions['year_month'] = df_sessions['date'].dt.to_period('M').astype(str)
monthly = df_sessions.groupby('year_month').agg(
    sessions=('date', 'count'),
    hours=('duration_min', lambda x: x.sum() / 60),
    avg_hr=('hr_avg', 'mean'),
).reset_index()

fig = px.bar(
    monthly,
    x='year_month',
    y='hours',
    title='Horas de entrenamiento por mes (2019–2026)',
    labels={'year_month': 'Mes', 'hours': 'Horas'},
)
fig.update_layout(xaxis_tickangle=-45, height=400)
fig.show()

In [15]:
# VO2max a lo largo del tiempo
fig = px.scatter(
    df_tests,
    x='date',
    y='vo2max_own_index',
    title='Evolución del VO2max (Polar OwnIndex)',
    labels={'date': 'Fecha', 'vo2max_own_index': 'OwnIndex (ml/kg/min)'},
    trendline='lowess',  # Línea de tendencia suavizada
)
fig.update_layout(height=400)
fig.show()

In [16]:
# HRV (RMSSD) a lo largo del tiempo — rolling 30 días
df_recovery_clean = df_recovery.dropna(subset=['rmssd']).copy()
df_recovery_clean['rmssd_30d'] = df_recovery_clean['rmssd'].rolling(30, min_periods=7).mean()

fig = px.line(
    df_recovery_clean,
    x='night',
    y='rmssd_30d',
    title='HRV (RMSSD) — Media móvil 30 días',
    labels={'night': 'Fecha', 'rmssd_30d': 'RMSSD (ms)'},
)
fig.update_layout(height=400)
fig.show()

---

## ¿Qué sigue?

Con los CSV guardados, el siguiente notebook (`02_exploratory_analysis.ipynb`) va a:

1. Analizar la distribución de intensidad por zonas de FC
2. Calcular tendencias de volumen y carga por año y fase
3. Cruzar entrenamiento con recovery y sueño
4. Identificar patrones por deporte

**Para subir esto a GitHub:**
1. Guarda este notebook (Ctrl+S)
2. Abre GitHub Desktop
3. Verás los archivos nuevos listados como 'Changes'
4. Escribe un mensaje de commit: "Add ETL pipeline — extract Polar data to CSV"
5. Click en 'Commit to main'
6. Click en 'Push origin'

¡Ese es tu primer commit profesional! 🎯

In [17]:
# Verificar que los CSV se guardaron correctamente
for filename in ['sessions.csv', 'fitness_tests.csv', 'nightly_recovery.csv', 'sleep_scores.csv']:
    filepath = os.path.join(PROCESSED_DIR, filename)
    if os.path.exists(filepath):
        size_kb = os.path.getsize(filepath) / 1024
        print(f'  ✓ {filename} — {size_kb:.0f} KB')
    else:
        print(f'  ✗ {filename} — NO ENCONTRADO')

  ✓ sessions.csv — 122 KB
  ✓ fitness_tests.csv — 1 KB
  ✓ nightly_recovery.csv — 55 KB
  ✓ sleep_scores.csv — 92 KB
